# 06 — NCCL / RoCE: GID, debug, collectives (L4+)

**Milestone:** GPUs can form a **process group** and exchange tensors over **RoCE** (not silently wedged on TCP or the wrong GID).

**FlashAttention is the victim, not the suspect.** A hang at “FlashAttention” in logs often means **distributed init / NCCL** is stuck—fix **HCA + GID + reachability** first.

See [`docs/troubleshooting-and-pitfalls.md`](../docs/troubleshooting-and-pitfalls.md) and [`docs/playbook-commands.md`](../docs/playbook-commands.md) for `NCCL_IB_HCA`, `NCCL_IB_GID_INDEX`, and **nccl-tests**.

## Checklist

- [ ] `show_gids` (or sysfs) used to pick **`NCCL_IB_GID_INDEX`** for **RoCE v2** on your device.
- [ ] `export NCCL_DEBUG=INFO` once in a controlled repro — confirm **interface** selection.
- [ ] Run **[NVIDIA/nccl-tests](https://github.com/NVIDIA/nccl-tests)** multi-node when in doubt ([`docs/playbook-commands.md`](../docs/playbook-commands.md) §5b).
- [ ] Symmetric **free VRAM** on both GPUs before TP jobs.

**Green signal:** NCCL logs show expected NIC/HCA; `all_reduce` benchmarks are in the right **order of magnitude** for a 200 **Gb/s-class** link (not stuck at “slow Ethernet” behavior).

In [ ]:
# ON SPARK — RoCE GID table (filter)
import subprocess
from pathlib import Path

IB_HINT = "rocep"  # broaden filter if needed

r = subprocess.run(["show_gids"], capture_output=True, text=True)
if r.returncode == 0:
    for line in r.stdout.splitlines():
        if IB_HINT in line or "v2" in line.lower():
            print(line)
else:
    print("show_gids failed:", r.stderr.strip() or r.stdout)
    print("\n/sys sample:")
    for p in sorted(Path("/sys/class/infiniband").glob("*/ports/*/gids/*"))[:8]:
        try:
            v = p.read_text().strip()
        except OSError:
            continue
        if v and v != "0000:0000:0000:0000:0000:0000:0000:0000":
            print(p, v)

## Next

[`07_vllm_tensor_parallel_launch_and_health.ipynb`](07_vllm_tensor_parallel_launch_and_health.ipynb) — **vLLM** + eugr launch + API smoke.